# ROI Loader Check

This notebook checks the ROI-region loader on:
- 5 local rendered-object samples
- 5 PSD samples

The primary ROI view uses an object-selection mask.
The last column stays as the Otsu residual mask so you can compare them directly.

For each sample it shows:
- glossy input
- diffuse target
- full residual
- residual masked by the selected ROI idea
- diffuse reconstructed from full residual
- diffuse reconstructed from the selected masked residual
- residual strength map
- object-selection mask
- Otsu residual mask

There is also a switch to compare the object-selection ROI against the older Otsu-masked idea.


In [ ]:
from pathlib import Path
from pprint import pprint
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_specdiff_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "Specular-Highlights" / "SpecDiff",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "Run_Training.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate SpecDiff/Run_Training.py from the current working directory.")


SPECDIFF_DIR = find_specdiff_dir()
PROJECT_ROOT = SPECDIFF_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
if str(SPECDIFF_DIR) not in sys.path:
    sys.path.insert(0, str(SPECDIFF_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import Run_Training as rt
from pipeline import ResidualLoader as residual_loader


print(f"SpecDiff dir: {SPECDIFF_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Workspace root: {WORKSPACE_ROOT}")


def chw_to_display(chw: torch.Tensor) -> np.ndarray:
    array = chw.detach().cpu().permute(1, 2, 0).float().numpy()
    return np.clip(array, 0.0, 1.0)


def residual_to_display(residual: torch.Tensor) -> np.ndarray:
    array = residual.detach().cpu().permute(1, 2, 0).float()
    array = array - array.min()
    array = array / (array.max() + 1e-8)
    return array.clamp(0.0, 1.0).numpy()


def mask_to_display(mask: torch.Tensor) -> np.ndarray:
    return mask.detach().cpu().squeeze().float().numpy()


def build_local_dataset(
    *,
    model_name: str,
    split: str,
    image_size: int,
    threshold_method: str,
    threshold: float,
    soft_gamma: float,
    kernel_size: int,
    fill_holes: bool,
    residual_mode: str,
):
    diffuse_dir = PROJECT_ROOT / "data" / split / model_name / "diffuse"
    glossy_dir = PROJECT_ROOT / "data" / split / model_name / "glossy"

    dataset = residual_loader.PSDResidualDataset(
        diffuse_dir=str(diffuse_dir),
        glossy_dir=str(glossy_dir),
        resize_hw=image_size,
        threshold_method=threshold_method,
        threshold=threshold,
        kernel_size=kernel_size,
        soft_gamma=soft_gamma,
        fill_holes=fill_holes,
        residual_mode=residual_mode,
    )
    info = {
        "dataset_source": "local_rendered",
        "model_name": model_name,
        "split": split,
        "glossy_dir": str(glossy_dir),
        "diffuse_dir": str(diffuse_dir),
        "dataset_size": len(dataset),
    }
    return dataset, info


def build_psd_dataset(
    *,
    psd_root,
    split: str,
    image_size: int,
    threshold_method: str,
    threshold: float,
    soft_gamma: float,
    kernel_size: int,
    fill_holes: bool,
    residual_mode: str,
):
    resolved_root = rt.resolve_psd_root(Path(psd_root).expanduser().resolve(), split=split)
    glossy_dir, diffuse_dir = rt.resolve_split_dirs(resolved_root, split)

    dataset = residual_loader.PSDResidualDataset(
        diffuse_dir=str(diffuse_dir),
        glossy_dir=str(glossy_dir),
        resize_hw=image_size,
        threshold_method=threshold_method,
        threshold=threshold,
        kernel_size=kernel_size,
        soft_gamma=soft_gamma,
        fill_holes=fill_holes,
        residual_mode=residual_mode,
    )
    info = {
        "dataset_source": "psd",
        "psd_root": str(resolved_root),
        "split": split,
        "glossy_dir": str(glossy_dir),
        "diffuse_dir": str(diffuse_dir),
        "dataset_size": len(dataset),
    }
    return dataset, info


def pick_indices(dataset_len: int, n_samples: int, seed: int) -> list[int]:
    n = min(n_samples, dataset_len)
    rng = random.Random(seed)
    return rng.sample(range(dataset_len), n)


def get_full_residual(sample: dict, residual_mode: str) -> torch.Tensor:
    key = f"unmasked_{residual_mode}_residual"
    return sample.get(key, sample["unmasked_residual"])


def compute_object_selection(sample: dict) -> torch.Tensor:
    glossy = sample["input"]
    diffuse = sample["diffuse"]
    glossy_luma = glossy.mean(dim=0)
    diffuse_luma = diffuse.mean(dim=0)
    positive_residual = (glossy_luma - diffuse_luma).clamp_min(0.0)

    object_support = torch.maximum(glossy_luma, diffuse_luma)
    object_threshold = torch.maximum(
        torch.tensor(0.02, device=object_support.device, dtype=object_support.dtype),
        object_support.max() * 0.05,
    )
    object_selection = object_support > object_threshold
    if not torch.any(object_selection):
        object_selection = positive_residual > 0

    return object_selection.float().unsqueeze(0)


def get_roi_mask(sample: dict, mask_kind: str) -> torch.Tensor:
    if mask_kind == "object_selection":
        return compute_object_selection(sample)
    if mask_kind == "otsu":
        return sample["mask"]
    raise ValueError("mask_kind must be 'object_selection' or 'otsu'.")


def get_masked_residual(sample: dict, residual_mode: str, mask_kind: str) -> torch.Tensor:
    full_residual = get_full_residual(sample, residual_mode)
    roi_mask = get_roi_mask(sample, mask_kind).expand_as(full_residual)
    return full_residual * roi_mask


def other_mask_kind(mask_kind: str) -> str:
    if mask_kind == "object_selection":
        return "otsu"
    if mask_kind == "otsu":
        return "object_selection"
    raise ValueError("mask_kind must be 'object_selection' or 'otsu'.")


def sample_metrics(sample: dict, residual_mode: str, mask_kind: str) -> dict[str, object]:
    full_residual = get_full_residual(sample, residual_mode)
    masked_residual = get_masked_residual(sample, residual_mode, mask_kind)
    roi_mask = get_roi_mask(sample, mask_kind).expand_as(masked_residual)
    outside_mask = 1.0 - roi_mask

    reconstructed_full = residual_loader.reconstruct_diffuse(
        sample["input"],
        full_residual,
        residual_mode=residual_mode,
    )
    reconstructed_masked = residual_loader.reconstruct_diffuse(
        sample["input"],
        masked_residual,
        residual_mode=residual_mode,
    )

    full_err = (reconstructed_full - sample["diffuse"]).abs()
    masked_err = (reconstructed_masked - sample["diffuse"]).abs()

    return {
        "name": sample.get("name", "unknown"),
        "mask_kind": mask_kind,
        "selected_mask_fraction": float(get_roi_mask(sample, mask_kind).mean()),
        "object_selection_fraction": float(compute_object_selection(sample).mean()),
        "otsu_mask_fraction": float(sample["mask"].mean()),
        "full_recon_max_err": float(full_err.max()),
        "masked_recon_inside_mask_max_err": float((masked_err * roi_mask).max()),
        "masked_recon_full_max_err": float(masked_err.max()),
        "masked_residual_abs_max_outside_mask": float((masked_residual.abs() * outside_mask).max()),
    }


def summarize_dataset(dataset, indices: list[int], residual_mode: str, mask_kind: str) -> list[dict[str, object]]:
    return [sample_metrics(dataset[idx], residual_mode, mask_kind) for idx in indices]


def show_mask_style_samples(dataset, label: str, indices: list[int], residual_mode: str, mask_kind: str):
    if not indices:
        raise ValueError("No indices were selected.")

    n = len(indices)
    fig, axes = plt.subplots(n, 9, figsize=(36, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for row, ds_idx in enumerate(indices):
        sample = dataset[ds_idx]
        metrics = sample_metrics(sample, residual_mode, mask_kind)
        full_residual = get_full_residual(sample, residual_mode)
        masked_residual = get_masked_residual(sample, residual_mode, mask_kind)
        reconstructed_full = residual_loader.reconstruct_diffuse(
            sample["input"],
            full_residual,
            residual_mode=residual_mode,
        )
        reconstructed_masked = residual_loader.reconstruct_diffuse(
            sample["input"],
            masked_residual,
            residual_mode=residual_mode,
        )
        object_selection = compute_object_selection(sample)
        otsu_mask = sample["mask"]
        masked_title = f"{residual_mode} residual\n({mask_kind.replace('_', ' ')} masked)"

        panels = [
            (f"{label} glossy\n{sample['name']}", sample["input"], "rgb"),
            ("Diffuse", sample["diffuse"], "rgb"),
            (f"{residual_mode} residual\n(full)", full_residual, "residual"),
            (masked_title, masked_residual, "residual"),
            ("Reconstructed diffuse\nfrom full residual", reconstructed_full, "rgb"),
            ("Reconstructed diffuse\nfrom masked residual", reconstructed_masked, "rgb"),
            ("Residual strength map", sample["soft_mask"], "mask"),
            ("Object selection mask", object_selection, "mask"),
            ("Otsu residual mask", otsu_mask, "mask"),
        ]

        for axis, (title, tensor, mode) in zip(axes[row], panels):
            if mode == "mask":
                axis.imshow(mask_to_display(tensor), cmap="gray", vmin=0.0, vmax=1.0)
            elif mode == "residual":
                axis.imshow(residual_to_display(tensor))
            else:
                axis.imshow(chw_to_display(tensor))
            axis.set_title(title)
            axis.axis("off")

        axes[row][0].set_ylabel(
            f"idx {ds_idx}\nmask={mask_kind}\nsel={metrics['selected_mask_fraction']:.3f}\ninside={metrics['masked_recon_inside_mask_max_err']:.3e}",
            rotation=0,
            ha="right",
            va="center",
            labelpad=70,
        )

    plt.suptitle(f"{label} ROI loader check | primary mask = {mask_kind}", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


def show_dataset_samples(
    dataset,
    label: str,
    indices: list[int],
    residual_mode: str,
    primary_mask_kind: str = "object_selection",
    compare_with_other_mask_kind: bool = False,
):
    show_mask_style_samples(dataset, label, indices, residual_mode, primary_mask_kind)
    if compare_with_other_mask_kind:
        show_mask_style_samples(dataset, label, indices, residual_mode, other_mask_kind(primary_mask_kind))


In [ ]:
# Settings
LOCAL_MODEL_NAME = "can"
LOCAL_SPLIT = "train"
PSD_ROOT = WORKSPACE_ROOT / "PSD_Dataset" / "PSD_Dataset"
PSD_SPLIT = "train"

IMAGE_SIZE = 128
NUM_SAMPLES = 5
SEED = 0

THRESHOLD_METHOD = "otsu"
THRESHOLD = 0.9
SOFT_GAMMA = 1.0
KERNEL_SIZE = 3
FILL_HOLES = True
RESIDUAL_MODE = "subtractive"

PRIMARY_MASK_KIND = "object_selection"  # "object_selection" or "otsu"
COMPARE_WITH_OTHER_MASK_KIND = True


In [ ]:
local_dataset, local_info = build_local_dataset(
    model_name=LOCAL_MODEL_NAME,
    split=LOCAL_SPLIT,
    image_size=IMAGE_SIZE,
    threshold_method=THRESHOLD_METHOD,
    threshold=THRESHOLD,
    soft_gamma=SOFT_GAMMA,
    kernel_size=KERNEL_SIZE,
    fill_holes=FILL_HOLES,
    residual_mode=RESIDUAL_MODE,
)

psd_dataset, psd_info = build_psd_dataset(
    psd_root=PSD_ROOT,
    split=PSD_SPLIT,
    image_size=IMAGE_SIZE,
    threshold_method=THRESHOLD_METHOD,
    threshold=THRESHOLD,
    soft_gamma=SOFT_GAMMA,
    kernel_size=KERNEL_SIZE,
    fill_holes=FILL_HOLES,
    residual_mode=RESIDUAL_MODE,
)

local_indices = pick_indices(len(local_dataset), NUM_SAMPLES, SEED)
psd_indices = pick_indices(len(psd_dataset), NUM_SAMPLES, SEED)

print("=== Local dataset info ===")
pprint(local_info)
print(f"selected indices: {local_indices}")
print()
print("=== PSD dataset info ===")
pprint(psd_info)
print(f"selected indices: {psd_indices}")
print()
print(f"primary mask kind: {PRIMARY_MASK_KIND}")
print(f"compare with other mask kind: {COMPARE_WITH_OTHER_MASK_KIND}")


In [ ]:
show_dataset_samples(
    local_dataset,
    label="Local",
    indices=local_indices,
    residual_mode=RESIDUAL_MODE,
    primary_mask_kind=PRIMARY_MASK_KIND,
    compare_with_other_mask_kind=COMPARE_WITH_OTHER_MASK_KIND,
)


In [ ]:
show_dataset_samples(
    psd_dataset,
    label="PSD",
    indices=psd_indices,
    residual_mode=RESIDUAL_MODE,
    primary_mask_kind=PRIMARY_MASK_KIND,
    compare_with_other_mask_kind=COMPARE_WITH_OTHER_MASK_KIND,
)


In [ ]:
print(f"=== Local sample metrics | mask kind = {PRIMARY_MASK_KIND} ===")
pprint(summarize_dataset(local_dataset, local_indices, RESIDUAL_MODE, PRIMARY_MASK_KIND))
print()
print(f"=== PSD sample metrics | mask kind = {PRIMARY_MASK_KIND} ===")
pprint(summarize_dataset(psd_dataset, psd_indices, RESIDUAL_MODE, PRIMARY_MASK_KIND))

if COMPARE_WITH_OTHER_MASK_KIND:
    alt_mask_kind = other_mask_kind(PRIMARY_MASK_KIND)
    print()
    print(f"=== Local sample metrics | mask kind = {alt_mask_kind} ===")
    pprint(summarize_dataset(local_dataset, local_indices, RESIDUAL_MODE, alt_mask_kind))
    print()
    print(f"=== PSD sample metrics | mask kind = {alt_mask_kind} ===")
    pprint(summarize_dataset(psd_dataset, psd_indices, RESIDUAL_MODE, alt_mask_kind))
